In [1]:
import json
import os.path
from collections.abc import Callable
from dataclasses import dataclass
from statistics import mean
from typing import Literal, NamedTuple

import numpy as np
import pandas as pd
from jaxtyping import Float
from scipy.stats import t

from tiu_phi_3_5_mini.data_management import load_train_validation_splits, ActivationsDataSelector, \
    train_scenarios_spec_path, ProbeTrainScenario
from tiu_phi_3_5_mini.direction_learning import ReconLosses
from tiu_phi_3_5_mini.evaluation_utils import MetricsForDatasetProbes, ConfusionMetrics, TraditionalMetrics
from tiu_phi_3_5_mini.logging_setup import create_logger
from tiu_phi_3_5_mini.phi_3_5_constants import dsets_index_path, \
    directions_reconstruction_losses_path, separation_by_layer_analysis_path, train_split_classification_metrics_path, \
    validation_split_classification_metrics_path, test_classification_metrics_path
from tiu_phi_3_5_mini.utils import FloatLikeT

In [ ]:
logger = create_logger(__name__)

In [2]:
dsets_index_df = pd.read_csv(dsets_index_path, index_col="Idx")
num_dsets = dsets_index_df.shape[0]

In [3]:
train_valid_split_specs = load_train_validation_splits()
data_selector = ActivationsDataSelector(dsets_index_df, train_valid_split_specs)

In [ ]:
logger.info(f"loading probe-training scenarios from file {train_scenarios_spec_path}")
with train_scenarios_spec_path.open("r") as f:
    raw_scenarios_specs = json.load(f)
train_scenarios = [ProbeTrainScenario(**raw_scenario_spec) for raw_scenario_spec in raw_scenarios_specs]

In [ ]:
with directions_reconstruction_losses_path.open("r") as f:
    raw_dir_recon_losses = json.load(f)
directions_reconstruction_losses: dict[tuple[int,...], list[ReconLosses]] = {
    tuple(map(int, scenario_id_str.split())) : [ReconLosses(**raw_recon_loss) for raw_recon_loss in raw_recon_losses_lst] for scenario_id_str, raw_recon_losses_lst in raw_dir_recon_losses.items()
}

In [87]:
num_standard_topics = dsets_index_df[dsets_index_df["is_other"] == False]["Categ_Folder"].nunique()
assert num_standard_topics == len(data_selector.dset_idxs_for_6way_topics)

In [7]:
#'scenarios' describe the different combinations of data that directions/probes were trained on 
other_categs: set[str] = set(dsets_index_df.loc[dsets_index_df.is_other].Categ_Folder.to_list())

scenario_standard_categs: dict[tuple[int,...], list[str]] = {}
scenario_other_dset_names: dict[tuple[int,...], list[str]] = {}
for scenario in train_scenarios:
    scenario_id = scenario.scenario_key()
    scenario_standard_categs[scenario_id] = []
    scenario_other_dset_names[scenario_id] = []
    for dset_idx_in_scenario in scenario_id:
        if not dsets_index_df.loc[dset_idx_in_scenario, "is_other"]:
            curr_standard_categ = dsets_index_df.loc[dset_idx_in_scenario, "Categ_Folder"]
            if curr_standard_categ not in scenario_standard_categs[scenario_id]:
                scenario_standard_categs[scenario_id].append(curr_standard_categ)
        else:
            curr_other_dset_name = os.path.splitext(dsets_index_df.loc[dset_idx_in_scenario, "Dataset_File"])[0]
            assert curr_other_dset_name not in scenario_other_dset_names[scenario_id]
            scenario_other_dset_names[scenario_id].append(curr_other_dset_name)

scenario_data_variants: dict[tuple[int,...], set[Literal["affirm", "neg", "conj", "disj", "other"]]] = {}
for scenario in train_scenarios:
    scenario_id = scenario.scenario_key()
    scenario_data_variants[scenario_id] = set()
    for dset_idx_in_scenario in scenario_id:
        assert not dsets_index_df.at[dset_idx_in_scenario, "in_german"]
        scenario_data_variants[scenario_id].add(
            "other" if dsets_index_df.at[dset_idx_in_scenario, "is_other"] else "neg" if dsets_index_df.at[dset_idx_in_scenario, "is_negated"] else "conj" if dsets_index_df.at[dset_idx_in_scenario, "is_conj"] else "disj" if dsets_index_df.at[dset_idx_in_scenario, "is_disj"] else "affirm")

scenario_labels: dict[tuple[int,...], str] = {scenario.scenario_key(): f"{scenario.result_folder_name}-{scenario.scenario_name}" for scenario in train_scenarios}

In [8]:
t_f_sep_by_layer_df = pd.read_csv(separation_by_layer_analysis_path, index_col="Idx")
ambig_truth_idx, ambig_lie_idx = data_selector.idxs_for_other_dsets["ambiguous_truthful_reply"], data_selector.idxs_for_other_dsets["ambiguous_lie"]
t_f_sep_by_layer_df.at[ambig_truth_idx, "Separation after 18"] = t_f_sep_by_layer_df.at[ambig_lie_idx, "Separation after 18"]
t_f_sep_by_layer_df.at[ambig_truth_idx, "Separation after 25"] = t_f_sep_by_layer_df.at[ambig_lie_idx, "Separation after 25"]
unambig_truth_idx, unambig_lie_idx = data_selector.idxs_for_other_dsets["unambiguous_truthful_reply"], data_selector.idxs_for_other_dsets["unambiguous_lie"]
t_f_sep_by_layer_df.at[unambig_truth_idx, "Separation after 18"] = t_f_sep_by_layer_df.at[unambig_lie_idx, "Separation after 18"]
t_f_sep_by_layer_df.at[unambig_truth_idx, "Separation after 25"] = t_f_sep_by_layer_df.at[unambig_lie_idx, "Separation after 25"]

In [9]:
with train_split_classification_metrics_path.open("r") as f:
    serialized_train_metrics = json.load(f)
with validation_split_classification_metrics_path.open("r") as f:
    serialized_validation_metrics = json.load(f)
train_metrics: dict[tuple[int,...], list[MetricsForDatasetProbes]] = {
    tuple(map(int, scenario_str.split(' '))): [MetricsForDatasetProbes.from_dict(metrics_dict) for metrics_dict in metrics_dicts_lst] 
    for scenario_str, metrics_dicts_lst in serialized_train_metrics.items()
}
validation_metrics: dict[tuple[int,...], list[MetricsForDatasetProbes]] = {
    tuple(map(int, scenario_str.split(' '))): [MetricsForDatasetProbes.from_dict(metrics_dict) for metrics_dict in metrics_dicts_lst] 
    for scenario_str, metrics_dicts_lst in serialized_validation_metrics.items()
}

In [10]:
with test_classification_metrics_path.open("r") as f:
    serialized_probes_metrics_on_test_dsets = json.load(f)
# first level key identifies the scenario (of one or more datasets) which the probe was trained on;
# second level key identifies the previously-unseen dataset which the probe is tested on
# third level index identifies which train-validation split was used to train a particular group of comparable probes 
#  for a scenario
probes_metrics_on_test_dsets: dict[tuple[int,...], dict[int, list[MetricsForDatasetProbes]]] = {
    tuple(map(int, scenario_str.split(' '))): {
        int(test_dset_idx): [MetricsForDatasetProbes.from_dict(serialized_test_metrics) for serialized_test_metrics in serialized_metrics_lst_for_test_dset]
        for test_dset_idx, serialized_metrics_lst_for_test_dset in serialized_test_dsets_metrics_for_scenario.items()
    } for scenario_str, serialized_test_dsets_metrics_for_scenario in serialized_probes_metrics_on_test_dsets.items()
}

In [ ]:
class ConfInterval(NamedTuple):
    low_b: FloatLikeT
    up_b: FloatLikeT

def calc_conf_interval(vals: Float[np.ndarray, "n"], alpha=0.05) -> ConfInterval:
    """
    calculates a confidence interval for the population mean of a random variable based on a sample of values
    :param vals: a sample of values for some random variable
    :param alpha: chance of incorrectly rejecting the null hypothesis, i.e. complement of confidence interval width
                    based on this parameter's default value, this will by default produce 95% confidence intervals 
    :return: a confidence interval for the population mean of a random variable
    """
    n = len(vals)
    degrees_of_freedom = n - 1
    t_critical = t.ppf(1-alpha/2, degrees_of_freedom)
    sample_mean, sample_std_dev = np.mean(vals), np.std(vals)
    conf_interval_margin_of_error = t_critical * sample_std_dev / np.sqrt(n)
    return ConfInterval(sample_mean - conf_interval_margin_of_error, sample_mean + conf_interval_margin_of_error)

In [11]:
class TruthPolarityDirsEval(NamedTuple):
    rel_loss_reducts_on_train: Float[np.ndarray, "n"]
    rel_loss_reducts_on_validation: Float[np.ndarray, "n"]
    rel_loss_reducts_on_validation_using_validation_mean: Float[np.ndarray, "n"]

lyr18_dirs_evals: dict[tuple[int,...], TruthPolarityDirsEval] = {}

for scenario_id, recon_losses_lst in directions_reconstruction_losses.items():
    rel_loss_reducts_on_train: list[float] = []
    rel_loss_reducts_on_validation: list[float] = []
    rel_loss_reducts_on_validation_using_validation_mean: list[float] = []
    
    for recon_losses in recon_losses_lst:
        rel_loss_reducts_on_train.append(1-recon_losses.train_mean_activ_and_t_p_dirs_loss_on_train/recon_losses.train_mean_activ_loss_on_train)
        rel_loss_reducts_on_validation.append(1-recon_losses.train_mean_activ_and_t_p_dirs_loss_on_validation/recon_losses.train_mean_activ_loss_on_validation)
        rel_loss_reducts_on_validation_using_validation_mean.append(1-recon_losses.validation_mean_activ_and_t_p_dirs_loss_on_validation/recon_losses.validation_mean_activ_loss_on_validation)
    
    lyr18_dirs_evals[scenario_id] = TruthPolarityDirsEval(
        rel_loss_reducts_on_train= np.asarray(rel_loss_reducts_on_train),
        rel_loss_reducts_on_validation= np.asarray(rel_loss_reducts_on_validation),
        rel_loss_reducts_on_validation_using_validation_mean= np.asarray(rel_loss_reducts_on_validation_using_validation_mean)
    )


In [12]:
@dataclass(frozen=True)
class ProbeEvalOnData:
    accs: Float[np.ndarray, "n"]
    f1s: Float[np.ndarray, "n"]
    briers: Float[np.ndarray, "n"]
    soft_f1s: Float[np.ndarray, "n"]

    @classmethod
    def combine(cls, *probe_evals_on_dsets: 'ProbeEvalOnData') -> 'ProbeEvalOnData':
        accs_lst: list[np.ndarray] = []
        f1s_lst: list[np.ndarray] = []
        briers_lst: list[np.ndarray] = []
        soft_f1s_lst: list[np.ndarray] = []
        
        for probe_eval_on_dset in probe_evals_on_dsets:
            accs_lst.append(probe_eval_on_dset.accs)
            f1s_lst.append(probe_eval_on_dset.f1s)
            briers_lst.append(probe_eval_on_dset.briers)
            soft_f1s_lst.append(probe_eval_on_dset.soft_f1s)
        
        return cls(np.concat(accs_lst), np.concat(f1s_lst), np.concat(briers_lst), np.concat(soft_f1s_lst))


def collect_probe_evals(probes_metrics_lst: list[MetricsForDatasetProbes]) -> tuple[ProbeEvalOnData, ProbeEvalOnData]:
    ttpd_accs: list[float] = []
    ttpd_f1s: list[float] = []
    ttpd_briers: list[float] = []
    ttpd_soft_f1s: list[float] = []
    
    baseline_accs: list[float] = []
    baseline_f1s: list[float] = []
    baseline_briers: list[float] = []
    baseline_soft_f1s: list[float] = []

    for probes_metrics in probes_metrics_lst:        
        ttpd_metrics = probes_metrics.lyr18_probe_metrics
        ttpd_trad_metrics = ttpd_metrics.get_traditional_metrics()
        ttpd_soft_metrics = ttpd_metrics.get_soft_metrics()
        
        ttpd_accs.append(ttpd_trad_metrics.accuracy)
        ttpd_f1s.append(ttpd_trad_metrics.f1)
        ttpd_briers.append(ttpd_metrics.get_brier_score())
        ttpd_soft_f1s.append(ttpd_soft_metrics.soft_f1)
        
        baseline_metrics = probes_metrics.lyr18_baseline_linear_probe_metrics
        baseline_trad_metrics = baseline_metrics.get_traditional_metrics()
        baseline_soft_metrics = baseline_metrics.get_soft_metrics()
        
        baseline_accs.append(baseline_trad_metrics.accuracy)
        baseline_f1s.append(baseline_trad_metrics.f1)
        baseline_briers.append(baseline_metrics.get_brier_score())
        baseline_soft_f1s.append(baseline_soft_metrics.soft_f1)
    
    ttpd_probe_eval = ProbeEvalOnData(
        accs=np.asarray(ttpd_accs), f1s=np.asarray(ttpd_f1s), briers=np.asarray(ttpd_briers), soft_f1s=np.asarray(ttpd_soft_f1s))
    
    baseline_probe_eval = ProbeEvalOnData(
        accs=np.asarray(baseline_accs), f1s=np.asarray(baseline_f1s), briers=np.asarray(baseline_briers), 
        soft_f1s=np.asarray(baseline_soft_f1s))
    
    return ttpd_probe_eval, baseline_probe_eval

ttpd_probes_train_evals: dict[tuple[int,...], ProbeEvalOnData] = {}
baseline_probes_train_evals: dict[tuple[int,...], ProbeEvalOnData] = {}

for scenario_id, probes_metrics_on_train_lst in train_metrics.items():        
    ttpd_train_eval, baseline_train_eval = collect_probe_evals(probes_metrics_on_train_lst)
    ttpd_probes_train_evals[scenario_id] = ttpd_train_eval
    baseline_probes_train_evals[scenario_id] = baseline_train_eval

ttpd_probes_val_evals: dict[tuple[int,...], ProbeEvalOnData] = {}
baseline_probes_val_evals: dict[tuple[int,...], ProbeEvalOnData] = {}

for scenario_id, probes_metrics_on_validation_lst in validation_metrics.items():
    ttpd_val_eval, baseline_val_eval = collect_probe_evals(probes_metrics_on_validation_lst)
    ttpd_probes_val_evals[scenario_id] = ttpd_val_eval
    baseline_probes_val_evals[scenario_id] = baseline_val_eval

In [13]:
lyr18_probe_row_dicts: list[dict[str, str | ConfInterval]] = []

lyr18_baseline_linear_probe_row_dicts: list[dict[str, str | ConfInterval]] = []

lyr18_ttpd_probe_to_baseline_linear_probe_row_dicts: list[dict[str, str | ConfInterval]] = []

for scenario in train_scenarios:
    scenario_id = scenario.scenario_key()
    lyr18_probe_row_dict = {
        "scenario": scenario_labels[scenario_id],
        "tf_sep": mean([t_f_sep_by_layer_df.at[dset_idx_in_scenario, "Separation after 18"] 
                        for dset_idx_in_scenario in scenario_id]),
        "train_recon_improv": calc_conf_interval(lyr18_dirs_evals[scenario_id].rel_loss_reducts_on_train),
        "val_recon_improv": calc_conf_interval(lyr18_dirs_evals[scenario_id].rel_loss_reducts_on_validation),
        "val_recon_improv_w_val_mean": calc_conf_interval(
            lyr18_dirs_evals[scenario_id].rel_loss_reducts_on_validation_using_validation_mean),
        "val_acc": calc_conf_interval(ttpd_probes_val_evals[scenario_id].accs),
        "val_f1": calc_conf_interval(ttpd_probes_val_evals[scenario_id].f1s),
        "val_brier": calc_conf_interval(ttpd_probes_val_evals[scenario_id].briers),
        "val_soft_f1": calc_conf_interval(ttpd_probes_val_evals[scenario_id].soft_f1s),
        "train_acc": calc_conf_interval(ttpd_probes_train_evals[scenario_id].accs),
        "train_f1": calc_conf_interval(ttpd_probes_train_evals[scenario_id].f1s),
        "train_brier": calc_conf_interval(ttpd_probes_train_evals[scenario_id].briers),
        "train_soft_f1": calc_conf_interval(ttpd_probes_train_evals[scenario_id].soft_f1s),
    }
    lyr18_probe_row_dicts.append(lyr18_probe_row_dict)
    
    lyr18_baseline_linear_probe_row_dict = {
        "scenario": scenario_labels[scenario_id],
        "val_acc": calc_conf_interval(baseline_probes_val_evals[scenario_id].accs),
        "val_f1": calc_conf_interval(baseline_probes_val_evals[scenario_id].f1s),
        "val_brier": calc_conf_interval(baseline_probes_val_evals[scenario_id].briers),
        "val_soft_f1": calc_conf_interval(baseline_probes_val_evals[scenario_id].soft_f1s),
        "train_acc": calc_conf_interval(baseline_probes_train_evals[scenario_id].accs),
        "train_f1": calc_conf_interval(baseline_probes_train_evals[scenario_id].f1s),
        "train_brier": calc_conf_interval(baseline_probes_train_evals[scenario_id].briers),
        "train_soft_f1": calc_conf_interval(baseline_probes_train_evals[scenario_id].soft_f1s),
    }
    lyr18_baseline_linear_probe_row_dicts.append(lyr18_baseline_linear_probe_row_dict)
    
    lyr18_ttpd_probe_to_baseline_linear_probe_row_dicts.append({
        "scenario": scenario_labels[scenario_id],
        "val_acc": calc_conf_interval(
            ttpd_probes_val_evals[scenario_id].accs - baseline_probes_val_evals[scenario_id].accs),
        "val_f1": calc_conf_interval(ttpd_probes_val_evals[scenario_id].f1s - baseline_probes_val_evals[scenario_id].f1s),
        "val_brier": calc_conf_interval(baseline_probes_val_evals[scenario_id].briers - ttpd_probes_val_evals[scenario_id].briers),
        "val_soft_f1": calc_conf_interval(ttpd_probes_val_evals[scenario_id].soft_f1s - baseline_probes_val_evals[scenario_id].soft_f1s),
        "train_acc": calc_conf_interval(ttpd_probes_train_evals[scenario_id].accs - baseline_probes_train_evals[scenario_id].accs),
        "train_f1": calc_conf_interval(ttpd_probes_train_evals[scenario_id].f1s - baseline_probes_train_evals[scenario_id].f1s),
        "train_brier": calc_conf_interval(baseline_probes_train_evals[scenario_id].briers - ttpd_probes_train_evals[scenario_id].briers),
        "train_soft_f1": calc_conf_interval(ttpd_probes_train_evals[scenario_id].soft_f1s - baseline_probes_train_evals[scenario_id].soft_f1s),
    })

lyr18_eval_except_test_df = pd.DataFrame(lyr18_probe_row_dicts)

lyr18_baseline_linear_probes_eval_except_test_df = pd.DataFrame(lyr18_baseline_linear_probe_row_dicts)

lyr18_ttpd_probe_to_baseline_linear_probe_eval_except_test_df = pd.DataFrame(lyr18_ttpd_probe_to_baseline_linear_probe_row_dicts)

For the "tf_sep" aka "true-false separation" column, and the ambiguous/unambiguous truth/lie (plus "honest reply despite incentive to lie") datasets,  
the value in the "ambiguous lie" and "ambiguous truthful reply" rows represent the separation between ambiguous lies and truths,  
the value in the "unambiguous lie" and "unambiguous truthful reply" rows represent the separation between unambiguous lies and truths,
and the 0.0 values for the "honest reply despite incentive to lie" row should be ignored.

When evaluating the below numbers, please note that brier scores are better if they are lower (unlike all other metrics here, which are better if they are higher)

In [14]:
lyr18_eval_except_test_df

,scenario,tf_sep,train_recon_improv,val_recon_improv,val_recon_improv_w_val_mean,val_acc,val_f1,val_brier,val_soft_f1,train_acc,train_f1,train_brier,train_soft_f1
0,animal_class__affirmative,0.873458,4.543415e-01,5.163863e-01,3.967561e-01,1.000000,1.000000,2.445273e-08,0.999943,0.992366,0.992908,8.771814e-03,0.989470
1,animal_class__conjunction,0.314961,2.398567e-01,2.314483e-01,2.357424e-01,0.890000,0.899083,6.859379e-02,0.876145,0.917500,0.920097,6.192252e-02,0.879679
2,animal_class__disjunction,0.029296,2.734168e-02,3.243016e-02,3.037365e-02,0.740000,0.697674,1.855859e-01,0.586721,0.717500,0.729017,1.874590e-01,0.632118
3,animal_class__negated,0.579465,3.598452e-01,4.085457e-01,3.995034e-01,1.000000,1.000000,6.577489e-04,0.995938,0.984733,0.984848,1.469421e-02,0.975943
4,cities__affirmative,1.690080,6.311205e-01,6.175942e-01,6.166543e-01,0.983333,0.982578,1.536337e-02,0.974531,0.983278,0.983526,1.203893e-02,0.978910
5,cities__conjunction,0.388754,2.749372e-01,2.993844e-01,2.982197e-01,0.960000,0.964286,3.396462e-02,0.934809,0.930718,0.933759,4.737542e-02,0.912977
6,cities__disjunction,0.058523,5.700467e-02,4.585128e-02,4.644103e-02,0.800000,0.795918,1.366856e-01,0.716584,0.855000,0.852041,1.151929e-01,0.742089
7,cities__negated,0.868476,4.610974e-01,4.793974e-01,4.679124e-01,0.990000,0.991150,8.395773e-03,0.984477,0.985786,0.985482,1.017570e-02,0.979065
8,element_symb__affirmative,0.577514,3.498783e-01,4.370834e-01,4.490662e-01,0.973684,0.975610,2.134477e-02,0.965086,0.905405,0.912500,6.313708e-02,0.913252
9,element_symb__conjunction,0.425471,3.073251e-01,2.748850e-01,2.742458e-01,0.880000,0.884615,7.882613e-02,0.853128,0.932500,0.940659,5.868069e-02,0.892681


TODO fresh commentary on previous cell

Meanwhile, note that here brier score deltas are calculated so that they are better (for TTPD probes) if they are higher (ideally greater than zero), as with the comparisons of values for other metrics

In [ ]:
lyr18_ttpd_probe_to_baseline_linear_probe_eval_except_test_df

TODO review and/or update this commentary  
It is slightly interesting that the simple linear probes seem to be doing markedly better within topic and data variant than the TTPD probes, given that Bürger et al found them generalizing similarly well.

In [ ]:
# Specifically standard scenarios containing datasets from within a single topic
def collect_probe_evals_of_scenarios_with_suffix(evals_dict: dict[tuple[int,...], ProbeEvalOnData], scenario_name_suffix: str) -> ProbeEvalOnData:
    scenarios_keys =  [key for key, label in scenario_labels.items() if label.endswith(scenario_name_suffix) and not label.startswith("cross_topic-")]
    scenarios_probe_evals = [evals_dict[scenario_key] for scenario_key in scenarios_keys]
    return ProbeEvalOnData.combine(scenarios_probe_evals)
    

In [ ]:
ttpd_affirm_neg_conj_disj_train_accs = collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_train_evals, "-affirm_neg_conj_disj").accs
ttpd_affirm_neg_conj_disj_val_accs = collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_val_evals, "-affirm_neg_conj_disj").accs

baseline_affirm_neg_conj_disj_train_accs = collect_probe_evals_of_scenarios_with_suffix(baseline_probes_train_evals, "-affirm_neg_conj_disj").accs
baseline_affirm_neg_conj_disj_val_accs = collect_probe_evals_of_scenarios_with_suffix(baseline_probes_val_evals, "-affirm_neg_conj_disj").accs

print("Confidence intervals for accuracy when trained on all four main data variants within a topic")
pd.DataFrame({
    "metric": ["train_acc", "val_acc"],
    "TTPD": [calc_conf_interval(ttpd_affirm_neg_conj_disj_train_accs), calc_conf_interval(ttpd_affirm_neg_conj_disj_val_accs)],
    "Baseline LR": [calc_conf_interval(baseline_affirm_neg_conj_disj_train_accs), calc_conf_interval(baseline_affirm_neg_conj_disj_val_accs)],
    "TTPD Advantage": [calc_conf_interval(ttpd_affirm_neg_conj_disj_train_accs - baseline_affirm_neg_conj_disj_train_accs), 
                       calc_conf_interval(ttpd_affirm_neg_conj_disj_val_accs - baseline_affirm_neg_conj_disj_val_accs)]
})

In [82]:
ttpd_affirm_neg_train_acc_drops = calc_conf_interval(
    ttpd_affirm_neg_conj_disj_train_accs - collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_train_evals, "-affirm_neg").accs)
ttpd_affirm_neg_val_acc_drops = calc_conf_interval(
    ttpd_affirm_neg_conj_disj_val_accs - collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_val_evals, "-affirm_neg").accs)
ttpd_neg_conj_train_acc_drops = calc_conf_interval(
    ttpd_affirm_neg_conj_disj_train_accs - collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_train_evals, "-neg_conj").accs)
ttpd_neg_conj_val_acc_drops = calc_conf_interval(
    ttpd_affirm_neg_conj_disj_val_accs - collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_val_evals, "-neg_conj").accs)
ttpd_neg_disj_train_acc_drops = calc_conf_interval(
    ttpd_affirm_neg_conj_disj_train_accs - collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_train_evals, "-neg_disj").accs)
ttpd_neg_disj_val_acc_drops = calc_conf_interval(
    ttpd_affirm_neg_conj_disj_val_accs - collect_probe_evals_of_scenarios_with_suffix(ttpd_probes_val_evals, "-neg_disj").accs)

baseline_affirm_neg_train_acc_drops = calc_conf_interval(
    baseline_affirm_neg_conj_disj_train_accs - collect_probe_evals_of_scenarios_with_suffix(baseline_probes_train_evals, "-affirm_neg").accs)
baseline_affirm_neg_val_acc_drops = calc_conf_interval(
    baseline_affirm_neg_conj_disj_val_accs - collect_probe_evals_of_scenarios_with_suffix(baseline_probes_val_evals, "-affirm_neg").accs)
baseline_neg_conj_train_acc_drops = calc_conf_interval(
    baseline_affirm_neg_conj_disj_train_accs - collect_probe_evals_of_scenarios_with_suffix(baseline_probes_train_evals, "-neg_conj").accs)
baseline_neg_conj_val_acc_drops = calc_conf_interval(
    baseline_affirm_neg_conj_disj_val_accs - collect_probe_evals_of_scenarios_with_suffix(baseline_probes_val_evals, "-neg_conj").accs)
baseline_neg_disj_train_acc_drops = calc_conf_interval(
    baseline_affirm_neg_conj_disj_train_accs - collect_probe_evals_of_scenarios_with_suffix(baseline_probes_train_evals, "-neg_disj").accs)
baseline_neg_disj_val_acc_drops = calc_conf_interval(
    baseline_affirm_neg_conj_disj_val_accs - collect_probe_evals_of_scenarios_with_suffix(baseline_probes_val_evals, "-neg_disj").accs)

print("Confidence intervals for decline in accuracy when trained on 2 of the main data variants within a topic (rather than all 4)")
pd.DataFrame({
    "scenario": ["affirmative+negated", "negated+conjunction", "negated+disjunction"],
    "ttpd_train_acc_loss": [ttpd_affirm_neg_train_acc_drops, ttpd_neg_conj_train_acc_drops, ttpd_neg_disj_train_acc_drops],
    "ttpd_val_acc_loss": [ttpd_affirm_neg_val_acc_drops, ttpd_neg_conj_val_acc_drops, ttpd_neg_disj_val_acc_drops],
    "baseline_train_acc_loss": [baseline_affirm_neg_train_acc_drops, baseline_neg_conj_train_acc_drops, baseline_neg_disj_train_acc_drops],
    "baseline_val_acc_loss": [baseline_affirm_neg_val_acc_drops, baseline_neg_conj_val_acc_drops, baseline_neg_disj_val_acc_drops],
})

,scenario,train_acc_loss,val_acc_loss
0,negated,-0.007461,0.026196
1,conjunction,0.030802,0.088120
2,disjunction,0.147588,0.164787
3,affirmative+negated,-0.006603,0.011639
4,affirmative+disjunction,0.106691,0.124164
5,negated+conjunction,0.070486,0.099852
6,affirmative+negated+conjunction+disjunction,0.145237,0.159423


TODO fresh commentary on previous 2 cells  
TODO explain why latter cell reports this relative thing to reduce how much variation in probe performance between
topics would make confidence intervals for the latter cell's calculations wider

In [ ]:
lyr18_eval_except_test_df['tf_sep'].describe()

In [101]:
lyr18_t_f_sep_and_t_p_dirs_correlations = lyr18_eval_except_test_df[['tf_sep', 'train_recon_improv', 'val_recon_improv', 'val_recon_improv_w_val_mean', "val_acc", "train_acc"]].applymap(lambda conf_interv: (conf_interv[0]+conf_interv[1])/2).corr()
lyr18_t_f_sep_and_t_p_dirs_correlations

,tf_sep,train_recon_improv,val_recon_improv,val_recon_improv_w_val_mean,val_acc,train_acc
tf_sep,1.000000,0.836358,0.828997,0.830357,0.682296,0.600867
train_recon_improv,0.836358,1.000000,0.990417,0.990186,0.764079,0.689926
val_recon_improv,0.828997,0.990417,1.000000,0.994927,0.758730,0.664601
val_recon_improv_w_val_mean,0.830357,0.990186,0.994927,1.000000,0.762526,0.670064
val_acc,0.682296,0.764079,0.758730,0.762526,1.000000,0.704551
train_acc,0.600867,0.689926,0.664601,0.670064,0.704551,1.000000


TODO fresh commentary for previous 2 cells

In [32]:
# TODO remove this and instead lean on MetricsForDatasetProbes.combine() plus the previously-defined collect_probe_evals()
ttpd_metrics_on_test_dsets: dict[tuple[int,...], dict[int, ConfusionMetrics]] = {
    scenario_id: {
        dset_idx_in_scenario: test_dset_metrics.lyr18_probe_metrics for dset_idx_in_scenario, test_dset_metrics in test_dsets_metrics.items()
    } 
    for scenario_id, test_dsets_metrics in probes_metrics_on_test_dsets.items()
}

In [95]:
# Just using accuracy because there are going to be _so_ many columns in this analysis's dataframe and examination of validation accuracy vs f1 vs soft-f1 for layer 18 probes showed that they were generally very similar for the standard topics' datasets (because those datasets were specifically constructed to be balanced between true and false statements) TODO confirm whether this line's commentary still looks right
class GeneralizationAccuracyWithinVsAcrossTopics(NamedTuple):
    affirm_within_topics: float
    affirm_across_topics: float
    neg_within_topics: float
    neg_across_topics: float
    conj_within_topics: float
    conj_across_topics: float
    disj_within_topics: float
    disj_across_topics: float
    de_affirm_within_topics: float
    de_affirm_across_topics: float
    de_neg_within_topics: float
    de_neg_across_topics: float

class TopicGeneralizationSummary(NamedTuple):
    avg_across_topic_penalty: float
    affirm_across_topic_penalty: float
    neg_across_topic_penalty: float
    conj_across_topic_penalty: float
    disj_across_topic_penalty: float
    de_affirm_across_topic_penalty: float
    de_neg_across_topic_penalty: float

# TODO rework everything in this cell based on the changes

lyr18_probe_generalization_within_vs_across_topics: dict[tuple[int,...], GeneralizationAccuracyWithinVsAcrossTopics] = {}
lyr18_probe_topic_generalization_summary: dict[tuple[int,...], TopicGeneralizationSummary] = {}

avg_cross_topic_acc_by_source_and_target_data_variants = pd.DataFrame({
    "source_data_variants": ["affirm+neg", "neg+conj", "neg+disj", "affirm+neg+conj+disj", "multitopic_affirm+neg+conj"],
    "affirm": [0.0]*9, "neg": [0.0]*9, "conj": [0.0]*9, "disj": [0.0]*9
})

for scenario_id, test_dsets_metrics in probes_metrics_on_test_dsets.items():
    curr_standard_categs = scenario_standard_categs[scenario_id]
    curr_data_variants = scenario_data_variants[scenario_id]
    if not curr_standard_categs:
        continue
    unseen_standard_categs = list(set(data_selector.dset_idxs_for_6way_topics.keys()) - set(curr_standard_categs))
    metrics_on_test_dsets = ttpd_metrics_on_test_dsets[scenario_id]
    
    affirm_within_topics_acc = np.nan
    if "affirm" not in curr_data_variants:
        same_topics_affirm_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["affirm"]] for topic_nm in curr_standard_categs
        ])
        affirm_within_topics_acc = same_topics_affirm_test_metrics.get_traditional_metrics().accuracy
    across_topics_affirm_test_metrics = ConfusionMetrics.combine(*[
        metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["affirm"]] for topic_nm in unseen_standard_categs
    ])
    affirm_across_topics_acc = across_topics_affirm_test_metrics.get_traditional_metrics().accuracy
    neg_within_topics_acc = np.nan
    if "neg" not in curr_data_variants:
        same_topics_neg_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["neg"]] for topic_nm in curr_standard_categs
        ])
        neg_within_topics_acc = same_topics_neg_test_metrics.get_traditional_metrics().accuracy
    across_topics_neg_test_metrics = ConfusionMetrics.combine(*[
        metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["neg"]] for topic_nm in unseen_standard_categs
    ])
    neg_across_topics_acc = across_topics_neg_test_metrics.get_traditional_metrics().accuracy
    conj_within_topics_acc = np.nan
    if "conj" not in curr_data_variants:
        same_topics_conj_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["conj"]] for topic_nm in curr_standard_categs
        ])
        conj_within_topics_acc = same_topics_conj_test_metrics.get_traditional_metrics().accuracy
    across_topics_conj_test_metrics = ConfusionMetrics.combine(*[
        metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["conj"]] for topic_nm in unseen_standard_categs
    ])
    conj_across_topics_acc = across_topics_conj_test_metrics.get_traditional_metrics().accuracy
    disj_within_topics_acc = np.nan
    if "disj" not in curr_data_variants:
        same_topics_disj_test_metrics = ConfusionMetrics.combine(*[
            metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["disj"]] for topic_nm in curr_standard_categs
        ])
        disj_within_topics_acc = same_topics_disj_test_metrics.get_traditional_metrics().accuracy
    across_topics_disj_test_metrics = ConfusionMetrics.combine(*[
        metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["disj"]] for topic_nm in unseen_standard_categs
    ])
    disj_across_topics_acc = across_topics_disj_test_metrics.get_traditional_metrics().accuracy
    assert "de_affirm" not in curr_data_variants
    same_topics_de_affirm_test_metrics = ConfusionMetrics.combine(*[
        metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["de_affirm"]] for topic_nm in curr_standard_categs
    ])
    de_affirm_within_topics_acc = same_topics_de_affirm_test_metrics.get_traditional_metrics().accuracy
    across_topics_de_affirm_test_metrics = ConfusionMetrics.combine(*[
        metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["de_affirm"]] for topic_nm in unseen_standard_categs
    ])
    de_affirm_across_topics_acc = across_topics_de_affirm_test_metrics.get_traditional_metrics().accuracy
    assert "de_neg" not in curr_data_variants
    same_topics_de_neg_test_metrics = ConfusionMetrics.combine(*[
        metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["de_neg"]] for topic_nm in curr_standard_categs
    ])
    de_neg_within_topics_acc = same_topics_de_neg_test_metrics.get_traditional_metrics().accuracy
    across_topics_de_neg_test_metrics = ConfusionMetrics.combine(*[
        metrics_on_test_dsets[data_selector.dset_idxs_for_6way_topics[topic_nm]["de_neg"]] for topic_nm in unseen_standard_categs
    ])
    de_neg_across_topics_acc = across_topics_de_neg_test_metrics.get_traditional_metrics().accuracy
    
    if "other" not in curr_data_variants:
        avg_cross_topic_acc_source_row_selector: pd.Series
        match curr_data_variants:
            case _ if curr_data_variants in [{"affirm"}, {"neg"}, {"conj"}, {"disj"}]:
                avg_cross_topic_acc_source_row_selector = avg_cross_topic_acc_by_source_and_target_data_variants["source_data_variants"] == str(list(curr_data_variants)[0])
            case _ if curr_data_variants == {"affirm", "neg"}:
                avg_cross_topic_acc_source_row_selector = avg_cross_topic_acc_by_source_and_target_data_variants["source_data_variants"] == "affirm+neg"
            case _ if curr_data_variants == {"affirm", "disj"}:
                avg_cross_topic_acc_source_row_selector = avg_cross_topic_acc_by_source_and_target_data_variants["source_data_variants"] == "affirm+disj"
            case _ if curr_data_variants == {"neg", "conj"}:
                avg_cross_topic_acc_source_row_selector = avg_cross_topic_acc_by_source_and_target_data_variants["source_data_variants"] == "neg+conj"
            case _ if curr_data_variants == {"affirm", "neg", "conj", "disj"}:
                avg_cross_topic_acc_source_row_selector = avg_cross_topic_acc_by_source_and_target_data_variants["source_data_variants"] == "affirm+neg+conj+disj"
            case _ if curr_data_variants == {"affirm", "neg", "conj"}:
                avg_cross_topic_acc_source_row_selector = avg_cross_topic_acc_by_source_and_target_data_variants["source_data_variants"] == "multitopic_affirm+neg+conj"
            case _:
                raise ValueError(f"invalid value {curr_data_variants} for curr_data_variants for scenario {scenario_id}")
        avg_cross_topic_acc_by_source_and_target_data_variants.loc[avg_cross_topic_acc_source_row_selector, "affirm"] += affirm_across_topics_acc
        avg_cross_topic_acc_by_source_and_target_data_variants.loc[avg_cross_topic_acc_source_row_selector, "neg"] += neg_across_topics_acc
        avg_cross_topic_acc_by_source_and_target_data_variants.loc[avg_cross_topic_acc_source_row_selector, "conj"] += conj_across_topics_acc
        avg_cross_topic_acc_by_source_and_target_data_variants.loc[avg_cross_topic_acc_source_row_selector, "disj"] += disj_across_topics_acc
    
    lyr18_probe_generalization_within_vs_across_topics[scenario_id] = GeneralizationAccuracyWithinVsAcrossTopics(
        affirm_within_topics=affirm_within_topics_acc, affirm_across_topics=affirm_across_topics_acc, neg_within_topics=neg_within_topics_acc,
        neg_across_topics=neg_across_topics_acc, conj_within_topics=conj_within_topics_acc, conj_across_topics=conj_across_topics_acc,
        disj_within_topics=disj_within_topics_acc, disj_across_topics=disj_across_topics_acc, de_affirm_within_topics=de_affirm_within_topics_acc,
        de_affirm_across_topics=de_affirm_across_topics_acc, de_neg_within_topics=de_neg_within_topics_acc, de_neg_across_topics=de_neg_across_topics_acc
    )
    
    affirm_across_topic_acc_penalty = affirm_within_topics_acc-affirm_across_topics_acc
    neg_across_topic_acc_penalty = neg_within_topics_acc-neg_across_topics_acc
    conj_across_topic_acc_penalty = conj_within_topics_acc-conj_across_topics_acc
    disj_across_topic_acc_penalty = disj_within_topics_acc-disj_across_topics_acc
    de_affirm_across_topic_acc_penalty = de_affirm_within_topics_acc-de_affirm_across_topics_acc
    de_neg_across_topic_acc_penalty = de_neg_within_topics_acc-de_neg_across_topics_acc
    avg_across_topic_acc_penalty = float(np.nanmean([affirm_across_topic_acc_penalty, neg_across_topic_acc_penalty, conj_across_topic_acc_penalty, disj_across_topic_acc_penalty, de_affirm_across_topic_acc_penalty, de_neg_across_topic_acc_penalty]))
    
    lyr18_probe_topic_generalization_summary[scenario_id] = TopicGeneralizationSummary(
        avg_across_topic_penalty=avg_across_topic_acc_penalty, affirm_across_topic_penalty=affirm_across_topic_acc_penalty, 
        neg_across_topic_penalty=neg_across_topic_acc_penalty, conj_across_topic_penalty=conj_across_topic_acc_penalty,
        disj_across_topic_penalty=disj_across_topic_acc_penalty, de_affirm_across_topic_penalty=de_affirm_across_topic_acc_penalty,
        de_neg_across_topic_penalty=de_neg_across_topic_acc_penalty
    )

avg_cross_topic_acc_by_source_and_target_data_variants.loc[~avg_cross_topic_acc_by_source_and_target_data_variants["source_data_variants"].str.startswith("multitopic_"), ["affirm", "neg", "conj", "disj"]] /= num_standard_topics


generalization_within_vs_across_topics_df = pd.DataFrame([
    { "scenario": scenario_labels[scenario_id], **generalization_info._asdict()  } for scenario_id, generalization_info in lyr18_probe_generalization_within_vs_across_topics.items()
])
topic_generalization_summary_df = pd.DataFrame([
    { "scenario": scenario_labels[scenario_id], **generalization_summary._asdict()  } for scenario_id, generalization_summary in lyr18_probe_topic_generalization_summary.items()
])

In [96]:
generalization_within_vs_across_topics_df

,scenario,affirm_within_topics,affirm_across_topics,neg_within_topics,neg_across_topics,conj_within_topics,conj_across_topics,disj_within_topics,disj_across_topics,de_affirm_within_topics,de_affirm_across_topics,de_neg_within_topics,de_neg_across_topics
0,animal_class__affirmative,NaN,0.840160,0.835366,0.629704,0.596000,0.648942,0.522000,0.592400,0.900000,0.688000,0.520000,0.473896
1,animal_class__conjunction,0.993902,0.829504,0.932927,0.645688,NaN,0.749285,0.502000,0.612400,0.880000,0.656000,0.540000,0.489960
2,animal_class__disjunction,0.829268,0.857476,0.487805,0.511156,0.882000,0.692396,NaN,0.598400,0.500000,0.664000,0.540000,0.550201
3,animal_class__negated,0.859756,0.877456,NaN,0.837829,0.560000,0.644940,0.498000,0.513600,0.760000,0.696000,0.860000,0.638554
4,cities__affirmative,NaN,0.683423,0.804813,0.511670,0.618825,0.578400,0.476000,0.492400,0.940000,0.600000,0.500000,0.481928
5,cities__conjunction,0.982620,0.792340,0.969251,0.551765,NaN,0.782400,0.518000,0.555200,0.980000,0.704000,0.840000,0.526104
6,cities__disjunction,0.951203,0.667265,0.514037,0.471574,0.640854,0.695200,NaN,0.668800,0.660000,0.568000,0.580000,0.497992
7,cities__negated,0.691845,0.597247,NaN,0.779174,0.502003,0.545600,0.484000,0.489200,0.640000,0.552000,1.000000,0.650602
8,element_symb__affirmative,NaN,0.903388,0.500000,0.527340,0.960000,0.802173,0.482000,0.506800,0.660000,0.852000,0.520000,0.546185
9,element_symb__conjunction,0.715054,0.693391,0.500000,0.525663,NaN,0.817896,0.696000,0.585600,0.740000,0.792000,0.520000,0.514056


In [97]:
topic_generalization_summary_df

,scenario,avg_across_topic_penalty,affirm_across_topic_penalty,neg_across_topic_penalty,conj_across_topic_penalty,disj_across_topic_penalty,de_affirm_across_topic_penalty,de_neg_across_topic_penalty
0,animal_class__affirmative,0.068085,NaN,0.205662,-0.052942,-0.070400,0.212000,0.046104
1,animal_class__conjunction,0.123056,0.164399,0.287239,NaN,-0.110400,0.224000,0.050040
2,animal_class__disjunction,-0.007231,-0.028208,-0.023351,0.189604,NaN,-0.164000,-0.010201
3,animal_class__negated,0.033441,-0.017700,NaN,-0.084940,-0.015600,0.064000,0.221446
4,cities__affirmative,0.135048,NaN,0.293143,0.040425,-0.016400,0.340000,0.018072
5,cities__conjunction,0.232092,0.190280,0.417486,NaN,-0.037200,0.276000,0.313896
6,cities__disjunction,0.089213,0.283938,0.042464,-0.054346,NaN,0.092000,0.082008
7,cities__negated,0.096640,0.094598,NaN,-0.043597,-0.005200,0.088000,0.349398
8,element_symb__affirmative,-0.022499,NaN,-0.027340,0.157827,-0.024800,-0.192000,-0.026185
9,element_symb__conjunction,0.012069,0.021662,-0.025663,NaN,0.110400,-0.052000,0.005944


In [98]:
(avg_topic_generalization_penalty_across_all_scenarios := topic_generalization_summary_df.avg_across_topic_penalty.mean().item())

0.028407982234862916

The cross-topic generalization penalty is surprisingly small (<3 percentage-points on average), so we will ignore the "already-seen topic vs fully-unseen topic" distinction for analysis beyond this point.

In passing, it is surprising that the 3-topic affirmative+negated+conjunction scenario had worse cross-topic generalization than the average scenario.

In [99]:
avg_cross_topic_acc_by_source_and_target_data_variants

,source_data_variants,affirm,neg,conj,disj
0,affirm,0.843616,0.610274,0.719804,0.558467
1,neg,0.512040,0.805444,0.529898,0.494600
2,conj,0.712324,0.564139,0.786394,0.558800
3,disj,0.689239,0.521776,0.663132,0.589800
4,affirm+neg,0.888125,0.871955,0.615243,0.566733
5,affirm+disj,0.851367,0.664223,0.698740,0.571067
6,neg+conj,0.709734,0.828780,0.671226,0.560933
7,affirm+neg+conj+disj,0.889481,0.788482,0.697400,0.605333
8,multitopic_affirm+neg+conj,0.976424,0.877210,0.845476,0.526000


Unsurprisingly, the best generalization to unseen disjunctive datasets was achieved when the directions and probe were trained at least partially on disjunctive data. One interesting thing is that the affirm+neg+conj+disj scenarios generalized better to unseen topics' disjunctive data better than the only-disjunctive scenarios, while the scenario trained on affirmative, negated, and conjunctive data from multiple topics generalized terribly to unseen topics' disjunctive data.

In [37]:
lyr18_baseline_linear_probe_metrics_on_test_dsets: dict[tuple[int,...], dict[int, ConfusionMetrics]] = {
    scenario_id: {
        dset_idx_in_scenario: test_dset_metrics.lyr18_baseline_linear_probe_metrics for dset_idx_in_scenario, test_dset_metrics in test_dsets_metrics.items()
    } 
    for scenario_id, test_dsets_metrics in probes_metrics_on_test_dsets.items()
}

In [38]:
#using more verbose phrasing in just this case to disambiguate from affirmative english texts in 'other' topics (real world scenarios, relative comparison, true false)
is_dset_en_affirm_in_std_topic: dict[int, bool] = (~(dsets_index_df.is_negated | dsets_index_df.is_conj | dsets_index_df.is_disj | dsets_index_df.is_other | dsets_index_df.in_german)).to_dict()
is_dset_en_negated: dict[int, bool] = (dsets_index_df.is_negated & ~dsets_index_df.in_german).to_dict()
is_dset_conj: dict[int, bool] = dsets_index_df.is_conj.to_dict()
is_dset_disj: dict[int, bool] = dsets_index_df.is_disj.to_dict()
is_dset_de_affirm: dict[int, bool] = (~dsets_index_df.is_negated & dsets_index_df.in_german).to_dict()
is_dset_de_negated: dict[int, bool] = (dsets_index_df.is_negated & dsets_index_df.in_german).to_dict()

In [39]:
dset_categ_lookup: dict[int, str] = dsets_index_df["Categ_Folder"].to_dict()

In [40]:
# TODO rewrite all of the next 4 cells to account for multiple splits
lyr18_probes_acc_on_unseen_multi_topic_data_groupings_row_dicts: list[dict[str, str | float]] = []
lyr18_baseline_linear_probes_acc_on_unseen_multi_topic_data_groupings_row_dicts: list[dict[str, str | float]] = []
lyr18_probes_acc_on_unseen_single_topic_data_groupings_row_dicts: list[dict[str, str|float]] = []
lyr18_baseline_linear_probes_acc_on_unseen_single_topic_data_groupings_row_dicts: list[dict[str, str|float]] = []

lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_multi_topic_data_groupings_row_dicts: list[dict[str, str | float]] = []
lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_single_topic_data_groupings_row_dicts: list[dict[str, str|float]] = []

In [41]:
def get_aggregate_trad_metrics(probe_dsets_metrics: dict[int, ConfusionMetrics], relevance_condition: Callable[[int], bool]) -> TraditionalMetrics:
    relev_dsets_metrics = [metrics for curr_dset_idx, metrics in probe_dsets_metrics.items() if relevance_condition(curr_dset_idx)]
    return ConfusionMetrics.combine(*relev_dsets_metrics).get_traditional_metrics()

In [42]:
for scenario in train_scenarios:
    scenario_id = scenario.scenario_key()
    ttpd_probe_metrics = ttpd_metrics_on_test_dsets[scenario_id]
    baseline_probe_metrics = lyr18_baseline_linear_probe_metrics_on_test_dsets[scenario_id]
    
    ttpd_all_unseen_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, lambda idx: True).accuracy
    baseline_all_unseen_acc = get_aggregate_trad_metrics(baseline_probe_metrics, lambda idx: True).accuracy
    
    ttpd_all_unseen_en_affirm_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, lambda idx: is_dset_en_affirm_in_std_topic[idx]).accuracy
    baseline_all_unseen_en_affirm_acc = get_aggregate_trad_metrics(baseline_probe_metrics, lambda idx: is_dset_en_affirm_in_std_topic[idx]).accuracy
    
    ttpd_all_unseen_en_neg_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, lambda idx: is_dset_en_negated[idx]).accuracy
    baseline_all_unseen_en_neg_acc = get_aggregate_trad_metrics(baseline_probe_metrics, lambda idx: is_dset_en_negated[idx]).accuracy
    
    ttpd_all_unseen_conj_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, lambda idx: is_dset_conj[idx]).accuracy
    baseline_all_unseen_conj_acc = get_aggregate_trad_metrics(baseline_probe_metrics, lambda idx: is_dset_conj[idx]).accuracy
    
    ttpd_all_unseen_disj_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, lambda idx: is_dset_disj[idx]).accuracy
    baseline_all_unseen_disj_acc = get_aggregate_trad_metrics(baseline_probe_metrics, lambda idx: is_dset_disj[idx]).accuracy
    
    ttpd_all_unseen_de_affirm_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, lambda idx: is_dset_de_affirm[idx]).accuracy
    baseline_all_unseen_de_affirm_acc = get_aggregate_trad_metrics(baseline_probe_metrics, lambda idx: is_dset_de_affirm[idx]).accuracy
    
    ttpd_all_unseen_de_neg_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, lambda idx: is_dset_de_negated[idx]).accuracy
    baseline_all_unseen_de_neg_acc = get_aggregate_trad_metrics(baseline_probe_metrics, lambda idx: is_dset_de_negated[idx]).accuracy

    animal_class_filter: Callable[[int], bool] = lambda idx: dset_categ_lookup[idx] == "animal_class"
    ttpd_all_unseen_animal_class_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, animal_class_filter).accuracy
    baseline_all_unseen_animal_class_acc = get_aggregate_trad_metrics(baseline_probe_metrics, animal_class_filter).accuracy
    
    cities_filter: Callable[[int], bool] = lambda idx: dset_categ_lookup[idx] == "cities"
    ttpd_all_unseen_cities_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, cities_filter).accuracy
    baseline_all_unseen_cities_acc = get_aggregate_trad_metrics(baseline_probe_metrics, cities_filter).accuracy
    
    element_symbol_filter: Callable[[int], bool] = lambda idx: dset_categ_lookup[idx] == "element_symb"
    ttpd_all_unseen_element_symbol_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, element_symbol_filter).accuracy
    baseline_all_unseen_element_symbol_acc = get_aggregate_trad_metrics(baseline_probe_metrics, element_symbol_filter).accuracy
    
    facts_filter: Callable[[int], bool] = lambda idx: dset_categ_lookup[idx] == "facts"
    ttpd_all_unseen_facts_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, facts_filter).accuracy
    baseline_all_unseen_facts_acc = get_aggregate_trad_metrics(baseline_probe_metrics, facts_filter).accuracy
    
    inventors_filter: Callable[[int], bool] = lambda idx: dset_categ_lookup[idx] == "inventors"
    ttpd_all_unseen_inventors_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, inventors_filter).accuracy
    baseline_all_unseen_inventors_acc = get_aggregate_trad_metrics(baseline_probe_metrics, inventors_filter).accuracy
    
    sp_en_trans_filter: Callable[[int], bool] = lambda idx: dset_categ_lookup[idx] == "sp_en_trans"
    ttpd_all_unseen_sp_en_trans_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, sp_en_trans_filter).accuracy
    baseline_all_unseen_sp_en_trans_acc = get_aggregate_trad_metrics(baseline_probe_metrics, sp_en_trans_filter).accuracy
    
    real_world_scenarios_filter: Callable[[int], bool] = lambda idx: dset_categ_lookup[idx] == "real_world_scenarios"
    ttpd_all_unseen_real_world_scenarios_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, real_world_scenarios_filter).accuracy
    baseline_all_unseen_real_world_scenarios_acc = get_aggregate_trad_metrics(baseline_probe_metrics, real_world_scenarios_filter).accuracy
    
    relative_comparison_filter: Callable[[int], bool] = lambda idx: dset_categ_lookup[idx] == "relative_comparison"
    ttpd_all_unseen_relative_comparison_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, relative_comparison_filter).accuracy
    baseline_all_unseen_relative_comparison_acc = get_aggregate_trad_metrics(baseline_probe_metrics, relative_comparison_filter).accuracy
    
    true_false_filter: Callable[[int], bool] = lambda idx: dset_categ_lookup[idx] == "true_false"
    ttpd_all_unseen_true_false_acc = get_aggregate_trad_metrics(ttpd_probe_metrics, true_false_filter).accuracy
    baseline_all_unseen_true_false_acc = get_aggregate_trad_metrics(baseline_probe_metrics, true_false_filter).accuracy
    
    ttpd_multi_topic_acc_row_dict = {
        "scenario": scenario_labels[scenario_id], "all": ttpd_all_unseen_acc, "en_affirm": ttpd_all_unseen_en_affirm_acc, "en_negated": ttpd_all_unseen_en_neg_acc,
        "conj": ttpd_all_unseen_conj_acc, "disj": ttpd_all_unseen_disj_acc, "de_affirm": ttpd_all_unseen_de_affirm_acc, "de_negated": ttpd_all_unseen_de_neg_acc
    }
    lyr18_probes_acc_on_unseen_multi_topic_data_groupings_row_dicts.append(ttpd_multi_topic_acc_row_dict)
    baseline_multi_topic_acc_row_dict = {
        "scenario": scenario_labels[scenario_id], "all": baseline_all_unseen_acc, "en_affirm": baseline_all_unseen_en_affirm_acc, 
        "en_negated": baseline_all_unseen_en_neg_acc, "conj": baseline_all_unseen_conj_acc, "disj": baseline_all_unseen_disj_acc, 
        "de_affirm": baseline_all_unseen_de_affirm_acc, "de_negated": baseline_all_unseen_de_neg_acc
    }
    lyr18_baseline_linear_probes_acc_on_unseen_multi_topic_data_groupings_row_dicts.append(baseline_multi_topic_acc_row_dict)
    lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_multi_topic_data_groupings_row_dicts.append({
        col_nm: (ttpd_multi_topic_acc_row_dict[col_nm] if col_nm == "scenario" else ttpd_multi_topic_acc_row_dict[col_nm] - baseline_multi_topic_acc_row_dict[col_nm]) for col_nm in ttpd_multi_topic_acc_row_dict.keys()
    })
    
    ttpd_single_topic_acc_row_dict = {
        "scenario": scenario_labels[scenario_id], "animal class": ttpd_all_unseen_animal_class_acc, "cities": ttpd_all_unseen_cities_acc,
        "element symbols": ttpd_all_unseen_element_symbol_acc, "facts": ttpd_all_unseen_facts_acc,
        "inventors": ttpd_all_unseen_inventors_acc, "spanish english translation": ttpd_all_unseen_sp_en_trans_acc, 
        "real world scenarios": ttpd_all_unseen_real_world_scenarios_acc, "relative comparison": ttpd_all_unseen_relative_comparison_acc,
        "true false": ttpd_all_unseen_true_false_acc
    }
    lyr18_probes_acc_on_unseen_single_topic_data_groupings_row_dicts.append(ttpd_single_topic_acc_row_dict)
    baseline_single_topic_acc_row_dict = {
        "scenario": scenario_labels[scenario_id],  "animal class": baseline_all_unseen_animal_class_acc, "cities": baseline_all_unseen_cities_acc,
        "element symbols": baseline_all_unseen_element_symbol_acc, "facts": baseline_all_unseen_facts_acc,
        "inventors": baseline_all_unseen_inventors_acc, "spanish english translation": baseline_all_unseen_sp_en_trans_acc, 
        "real world scenarios": baseline_all_unseen_real_world_scenarios_acc, "relative comparison": baseline_all_unseen_relative_comparison_acc,
        "true false": baseline_all_unseen_true_false_acc
    }
    lyr18_baseline_linear_probes_acc_on_unseen_single_topic_data_groupings_row_dicts.append(baseline_single_topic_acc_row_dict)
    lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_single_topic_data_groupings_row_dicts.append({
        col_nm: (ttpd_single_topic_acc_row_dict[col_nm] if col_nm == "scenario" else ttpd_single_topic_acc_row_dict[col_nm] - baseline_single_topic_acc_row_dict[col_nm]) for col_nm in ttpd_single_topic_acc_row_dict.keys()
    })

In [43]:
lyr18_probes_acc_on_unseen_multi_topic_data_groupings_df = pd.DataFrame(lyr18_probes_acc_on_unseen_multi_topic_data_groupings_row_dicts)
lyr18_baseline_linear_probes_acc_on_unseen_multi_topic_data_groupings_df = pd.DataFrame(lyr18_baseline_linear_probes_acc_on_unseen_multi_topic_data_groupings_row_dicts)
lyr18_probes_acc_on_unseen_single_topic_data_groupings_df = pd.DataFrame(lyr18_probes_acc_on_unseen_single_topic_data_groupings_row_dicts)
lyr18_baseline_linear_probes_acc_on_unseen_single_topic_data_groupings_df = pd.DataFrame(lyr18_baseline_linear_probes_acc_on_unseen_single_topic_data_groupings_row_dicts)

lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_multi_topic_data_groupings_df = pd.DataFrame(lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_multi_topic_data_groupings_row_dicts)
lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_single_topic_data_groupings_df = pd.DataFrame(lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_single_topic_data_groupings_row_dicts)

In [44]:
lyr18_probes_acc_on_unseen_multi_topic_data_groupings_df

,scenario,all,en_affirm,en_negated,conj,disj,de_affirm,de_negated
0,animal_class__affirmative,0.598928,0.840160,0.640354,0.642321,0.580667,0.723333,0.481605
1,animal_class__conjunction,0.616058,0.838017,0.660562,0.749285,0.594000,0.693333,0.498328
2,animal_class__disjunction,0.646416,0.856015,0.509946,0.716108,0.598400,0.636667,0.548495
3,animal_class__negated,0.622431,0.876539,0.837829,0.634317,0.511000,0.706667,0.675585
4,cities__affirmative,0.573471,0.683423,0.650142,0.593547,0.489667,0.656667,0.484950
5,cities__conjunction,0.679085,0.882223,0.748974,0.782400,0.549000,0.750000,0.578595
6,cities__disjunction,0.615186,0.801389,0.491632,0.674837,0.668800,0.583333,0.511706
7,cities__negated,0.556986,0.641932,0.779174,0.529265,0.488333,0.566667,0.709030
8,element_symb__affirmative,0.701161,0.903388,0.525734,0.821911,0.502667,0.820000,0.541806
9,element_symb__conjunction,0.628308,0.694664,0.524155,0.817896,0.604000,0.783333,0.515050


In [45]:
lyr18_probes_acc_on_unseen_multi_topic_data_groupings_df.describe()

,all,en_affirm,en_negated,conj,disj,de_affirm,de_negated
count,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000
mean,0.618207,0.765337,0.697965,0.669601,0.555470,0.686444,0.606633
std,0.069084,0.181112,0.149788,0.119410,0.053205,0.116672,0.117852
min,0.465199,0.211241,0.491632,0.455978,0.476000,0.396667,0.438127
25%,0.569092,0.629697,0.548232,0.579794,0.504133,0.603333,0.510870
50%,0.624510,0.845174,0.683612,0.670669,0.548900,0.728333,0.561873
75%,0.672858,0.904814,0.851086,0.757652,0.598967,0.771667,0.713211
max,0.789570,0.976916,0.954720,0.883942,0.668800,0.843333,0.849498


In [46]:
lyr18_baseline_linear_probes_acc_on_unseen_multi_topic_data_groupings_df

,scenario,all,en_affirm,en_negated,conj,disj,de_affirm,de_negated
0,animal_class__affirmative,0.628325,0.864469,0.692138,0.674837,0.647667,0.730000,0.478261
1,animal_class__conjunction,0.727648,0.920429,0.735081,0.885363,0.515667,0.760000,0.628763
2,animal_class__disjunction,0.512028,0.525103,0.508052,0.514757,0.646000,0.526667,0.521739
3,animal_class__negated,0.663155,0.799179,0.864469,0.663332,0.504667,0.716667,0.782609
4,cities__affirmative,0.631684,0.763615,0.522577,0.685093,0.521333,0.740000,0.347826
5,cities__conjunction,0.705427,0.916009,0.520366,0.794800,0.577667,0.800000,0.468227
6,cities__disjunction,0.628012,0.805494,0.578781,0.564032,0.729600,0.720000,0.525084
7,cities__negated,0.633931,0.848121,0.828845,0.592796,0.541333,0.786667,0.789298
8,element_symb__affirmative,0.583103,0.796713,0.633723,0.571786,0.502000,0.763333,0.515050
9,element_symb__conjunction,0.680610,0.929586,0.500789,0.842767,0.530667,0.820000,0.461538


In [47]:
lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_multi_topic_data_groupings_df

,scenario,all,en_affirm,en_negated,conj,disj,de_affirm,de_negated
0,animal_class__affirmative,-0.029397,-0.024309,-0.051784,-0.032516,-0.067000,-0.006667,0.003344
1,animal_class__conjunction,-0.111591,-0.082412,-0.074518,-0.136078,0.078333,-0.066667,-0.130435
2,animal_class__disjunction,0.134387,0.330913,0.001895,0.201351,-0.047600,0.110000,0.026756
3,animal_class__negated,-0.040724,0.077360,-0.026640,-0.029015,0.006333,-0.010000,-0.107023
4,cities__affirmative,-0.058214,-0.080192,0.127566,-0.091546,-0.031667,-0.083333,0.137124
5,cities__conjunction,-0.026342,-0.033786,0.228608,-0.012400,-0.028667,-0.050000,0.110368
6,cities__disjunction,-0.012825,-0.004105,-0.087149,0.110805,-0.060800,-0.136667,-0.013378
7,cities__negated,-0.076945,-0.206189,-0.049671,-0.063532,-0.053000,-0.220000,-0.080268
8,element_symb__affirmative,0.118058,0.106676,-0.107989,0.250125,0.000667,0.056667,0.026756
9,element_symb__conjunction,-0.052302,-0.234923,0.023366,-0.024871,0.073333,-0.036667,0.053512


In [48]:
lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_multi_topic_data_groupings_df.describe()

,all,en_affirm,en_negated,conj,disj,de_affirm,de_negated
count,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000
mean,-0.027044,-0.004525,0.004781,-0.044980,-0.030094,-0.026000,-0.015886
std,0.065080,0.148862,0.124110,0.141573,0.066574,0.088112,0.086101
min,-0.167563,-0.398800,-0.401304,-0.421670,-0.185600,-0.293333,-0.230769
25%,-0.077052,-0.079252,-0.041887,-0.135649,-0.080800,-0.058333,-0.077759
50%,-0.032903,-0.004263,-0.008574,-0.033017,-0.022500,-0.030000,-0.008361
75%,0.009406,0.081860,0.044101,0.015195,0.009750,0.019167,0.027592
max,0.134387,0.330913,0.359015,0.355428,0.089000,0.200000,0.210702


In [49]:
lyr18_probes_acc_on_unseen_single_topic_data_groupings_df

,scenario,animal class,cities,element symbols,facts,inventors,spanish english translation,real world scenarios,relative comparison,true false
0,animal_class__affirmative,0.606804,0.792731,0.643342,0.600180,0.540795,0.611726,0.688742,0.621970,0.569232
1,animal_class__conjunction,0.687500,0.845187,0.626359,0.629446,0.565900,0.643805,0.562914,0.659343,0.577745
2,animal_class__disjunction,0.764009,0.736149,0.671196,0.567762,0.581590,0.676438,0.649007,0.780051,0.622041
3,animal_class__negated,0.594146,0.821022,0.622962,0.674471,0.620816,0.687500,0.562914,0.511616,0.601609
4,cities__affirmative,0.633754,0.679188,0.569293,0.579919,0.530335,0.497235,0.589404,0.543939,0.569644
5,cities__conjunction,0.783613,0.910356,0.664402,0.651959,0.584205,0.683075,0.629139,0.842172,0.641676
6,cities__disjunction,0.637955,0.700218,0.730299,0.568663,0.600941,0.653208,0.543046,0.566162,0.606250
7,cities__negated,0.525210,0.587368,0.537364,0.601081,0.617155,0.625553,0.529801,0.500000,0.553084
8,element_symb__affirmative,0.714986,0.722593,0.678849,0.675371,0.601987,0.739491,0.735099,0.758081,0.696957
9,element_symb__conjunction,0.701681,0.657760,0.655350,0.667717,0.581067,0.741704,0.582781,0.883586,0.587466


In [50]:
lyr18_probes_acc_on_unseen_single_topic_data_groupings_df.describe()

,animal class,cities,element symbols,facts,inventors,spanish english translation,real world scenarios,relative comparison,true false
count,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000
mean,0.636906,0.739048,0.641814,0.614063,0.598574,0.653819,0.636446,0.649268,0.596777
std,0.094262,0.132850,0.091259,0.068886,0.058142,0.101841,0.115169,0.137284,0.062378
min,0.467787,0.413949,0.384137,0.484466,0.453452,0.439712,0.443709,0.457576,0.462624
25%,0.563680,0.652407,0.568945,0.561830,0.548117,0.578125,0.541391,0.523295,0.558906
50%,0.636905,0.748134,0.670177,0.617731,0.599232,0.673949,0.628295,0.619823,0.602159
75%,0.708683,0.848183,0.706012,0.662877,0.648902,0.731471,0.720199,0.759912,0.640811
max,0.829832,0.970000,0.778533,0.786207,0.710000,0.838496,0.847682,0.941162,0.745169


In [51]:
lyr18_baseline_linear_probes_acc_on_unseen_single_topic_data_groupings_df

,scenario,animal class,cities,element symbols,facts,inventors,spanish english translation,real world scenarios,relative comparison,true false
0,animal_class__affirmative,0.704905,0.834185,0.675272,0.618640,0.559100,0.673119,0.596026,0.614899,0.598588
1,animal_class__conjunction,0.705819,0.892731,0.693614,0.706439,0.627092,0.727323,0.728477,0.903535,0.693964
2,animal_class__disjunction,0.591595,0.527505,0.527174,0.559658,0.548640,0.529314,0.443709,0.500000,0.503131
3,animal_class__negated,0.572785,0.852652,0.680027,0.611887,0.619770,0.626659,0.569536,0.854040,0.625968
4,cities__affirmative,0.636555,0.682248,0.611413,0.578118,0.514121,0.570796,0.741722,0.546970,0.648542
5,cities__conjunction,0.714286,0.710468,0.740489,0.688429,0.610879,0.713496,0.801325,0.938636,0.683007
6,cities__disjunction,0.681373,0.738562,0.694973,0.569113,0.603033,0.571350,0.642384,0.526263,0.628000
7,cities__negated,0.588936,0.682805,0.753397,0.618640,0.674686,0.812500,0.768212,0.504293,0.629511
8,element_symb__affirmative,0.648459,0.652849,0.602644,0.651959,0.506799,0.617257,0.701987,0.571970,0.568930
9,element_symb__conjunction,0.719188,0.778585,0.588477,0.648357,0.622385,0.698009,0.596026,0.594697,0.681716


In [52]:
lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_single_topic_data_groupings_df

,scenario,animal class,cities,element symbols,facts,inventors,spanish english translation,real world scenarios,relative comparison,true false
0,animal_class__affirmative,-0.098101,-0.041454,-0.031929,-0.018460,-0.018305,-0.061394,0.092715,0.007071,-0.029357
1,animal_class__conjunction,-0.018319,-0.047544,-0.067255,-0.076992,-0.061192,-0.083518,-0.165563,-0.244192,-0.116219
2,animal_class__disjunction,0.172414,0.208644,0.144022,0.008104,0.032950,0.147124,0.205298,0.280051,0.118910
3,animal_class__negated,0.021361,-0.031631,-0.057065,0.062584,0.001046,0.060841,-0.006623,-0.342424,-0.024359
4,cities__affirmative,-0.002801,-0.003061,-0.042120,0.001801,0.016213,-0.073562,-0.152318,-0.003030,-0.078898
5,cities__conjunction,0.069328,0.199889,-0.076087,-0.036470,-0.026674,-0.030420,-0.172185,-0.096465,-0.041330
6,cities__disjunction,-0.043417,-0.038344,0.035326,-0.000450,-0.002092,0.081858,-0.099338,0.039899,-0.021750
7,cities__negated,-0.063725,-0.095437,-0.216033,-0.017560,-0.057531,-0.186947,-0.238411,-0.004293,-0.076427
8,element_symb__affirmative,0.066527,0.069745,0.076205,0.023413,0.095188,0.122235,0.033113,0.186111,0.128028
9,element_symb__conjunction,-0.017507,-0.120825,0.066872,0.019361,-0.041318,0.043695,-0.013245,0.288889,-0.094249


In [53]:
lyr18_ttpd_probes_rel_to_baseline_probes_acc_delta_on_unseen_single_topic_data_groupings_df.describe()

,animal class,cities,element symbols,facts,inventors,spanish english translation,real world scenarios,relative comparison,true false
count,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000
mean,-0.039523,-0.002070,-0.035101,-0.014271,-0.015457,-0.031605,-0.008872,-0.046650,-0.027143
std,0.092275,0.117435,0.087600,0.058013,0.069418,0.100697,0.159499,0.160499,0.066390
min,-0.255602,-0.299411,-0.240741,-0.167942,-0.200000,-0.300332,-0.357616,-0.355808,-0.136211
25%,-0.096116,-0.059774,-0.093784,-0.053694,-0.048509,-0.089463,-0.145695,-0.177083,-0.077704
50%,-0.024510,0.001572,-0.041101,-0.004953,-0.021182,-0.036228,0.006623,-0.003662,-0.030483
75%,0.018032,0.067976,0.014096,0.026902,0.011343,0.034154,0.092715,0.072664,0.012914
max,0.172414,0.255206,0.229620,0.099505,0.157224,0.187500,0.450980,0.288889,0.128028
